In [1]:
import os
import scanpy as sc
import numpy as np
from vitessce.data_utils import (
    to_diamond,
    rgb_img_to_ome_zarr,
    optimize_adata,
    to_uint8
)

/usr/local/lib/python3.10/dist-packages/vitessce/__init__.py:42: UserWarning: Extra installs are necessary to use widgets: No module named 'anywidget'
  warn(f'Extra installs are necessary to use widgets: {e}')
/usr/local/lib/python3.10/dist-packages/vitessce/__init__.py:68: UserWarning: Extra installs are necessary to use exports: No module named 'starlette'
  warn(f'Extra installs are necessary to use exports: {e}')


In [2]:
# Load in the SRT anndata
all_samples = sc.read_h5ad("/zata/zippy/kresgeb/hippocampus/my_output/nmf/HPC/srt.h5ad")
all_samples

AnnData object with n_obs × n_vars = 150917 × 31483
    obs: 'sample_id', 'in_tissue', 'array_row', 'array_col', '10x_graphclust', '10x_kmeans_10_clusters', '10x_kmeans_2_clusters', '10x_kmeans_3_clusters', '10x_kmeans_4_clusters', '10x_kmeans_5_clusters', '10x_kmeans_6_clusters', '10x_kmeans_7_clusters', '10x_kmeans_8_clusters', '10x_kmeans_9_clusters', 'key', 'sum_umi', 'sum_gene', 'expr_chrM', 'expr_chrM_ratio', 'ManualAnnotation', 'brnum', 'dateImg', 'experimenterImg', 'slide', 'array', 'position', 'seqNum', 'experimenterSeq', 'dx', 'race', 'sex', 'age', 'pmi', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'total', 'low_sum_id', 'low_sum_br', 'low_detected_id', 'low_detected_br', 'discard_auto_br', 'discard_auto_id', 'sizeFactor', 'cluster', 'neuron_cell_body', 'domain', 'broad.domain', 'nmf1', 'nmf2', 'nmf3', 'nmf4', 'nmf5', 'nmf6', 'nmf7', 'nmf8', 'nmf9', 'nmf10', 'nmf11', 'nmf12', 'nmf13', 'nmf14', 'nmf15', 'nmf16', 'nmf17', 'nmf18', 'nm

In [ ]:
def split_by_sample(adata):
    """
    Split an AnnData object into a dictionary of subsets by sample_id.

    Args:
        adata (AnnData): The full AnnData object containing multiple samples with a 'sample_id' column in `.obs`.

    Returns:
        sample_dict (dict of str: Anndata): A dictionary mapping each unique sample_id (str) to a copied AnnData subset.

    Notes:
        Each subset is a copy, so modifying it does not affect the original AnnData.
        Progress is printed for each sample split.
    """
    sample_dict = {}
    sample_ids = adata.obs["sample_id"].unique()
    num_samples = len(sample_ids)
    for i, sample_id in enumerate(sample_ids, start=1):
        # Subset and copy
        sample_dict[sample_id] = adata[adata.obs["sample_id"] == sample_id].copy()
        # Drop these rows from adata
        adata = adata[adata.obs["sample_id"] != sample_id]
        print(f"Split {i}/{num_samples} samples ({i/num_samples:.2%} done)")

    return sample_dict

sample_dict = split_by_sample(all_samples)


Split 1/36 samples (2.78% done)
Split 2/36 samples (5.56% done)
Split 3/36 samples (8.33% done)
Split 4/36 samples (11.11% done)
Split 5/36 samples (13.89% done)
Split 6/36 samples (16.67% done)
Split 7/36 samples (19.44% done)
Split 8/36 samples (22.22% done)
Split 9/36 samples (25.00% done)
Split 10/36 samples (27.78% done)
Split 11/36 samples (30.56% done)
Split 12/36 samples (33.33% done)
Split 13/36 samples (36.11% done)
Split 14/36 samples (38.89% done)
Split 15/36 samples (41.67% done)
Split 16/36 samples (44.44% done)
Split 17/36 samples (47.22% done)
Split 18/36 samples (50.00% done)
Split 19/36 samples (52.78% done)
Split 20/36 samples (55.56% done)
Split 21/36 samples (58.33% done)
Split 22/36 samples (61.11% done)
Split 23/36 samples (63.89% done)
Split 24/36 samples (66.67% done)
Split 25/36 samples (69.44% done)
Split 26/36 samples (72.22% done)
Split 27/36 samples (75.00% done)
Split 28/36 samples (77.78% done)
Split 29/36 samples (80.56% done)
Split 30/36 samples (83.33

In [4]:
def scale_spatial_coords(adata, sample_id):
    """
    Scale the spatial coordinates of a single AnnData object by its image scale factor.

    Args:
        adata (AnnData): AnnData object containing spatial coordinates in `.obsm["spatial"]`.
        sample_id (str): The sample_id corresponding to this AnnData, used to locate the scale factor 
                         in `.uns["spatial"][sample_id]["scalefactors"]["tissue_hires_scalef"]`.

    Notes:
        Modifies the `.obsm["spatial"]` field of `adata` in place.
    """
    # Get scale factor for this sample
    scale_factor = adata.uns["spatial"][sample_id]["scalefactors"]["tissue_hires_scalef"]
    # Overwrite with scaled coordinates
    adata.obsm["spatial"] = adata.obsm["spatial"] * scale_factor


In [5]:
def add_segmentations(adata, radius=7):
    """
    Add diamond-shaped segmentations to an AnnData object using spatial coordinates.

    Args:
        adata (AnnData): AnnData object with `.obsm["spatial"]` coordinates.
        radius (int): Radius for diamond shape.

    Notes:
        Modifies the `.obsm["segmentations"]` field of `adata` in place.
    """
    spatial = adata.obsm["spatial"]
    segmentations = [to_diamond(x, y, radius) for x, y in spatial]
    adata.obsm["segmentations"] = np.array(segmentations)


In [6]:
def write_ome_zarr_image(adata, sample_id, output_path):
    """
    Write the high-resolution RGB image from an AnnData object to OME-Zarr format.

    Args:
        adata (AnnData): AnnData object containing the high-resolution image at 
            `.uns["spatial"][sample_id]["images"]["hires"]`.
        sample_id (str): Key for the sample in `.uns["spatial"]`.
        output_path (str): Destination path for the OME-Zarr output (e.g., "image.ome.zarr").
        

    Raises:
        ValueError: If the high-resolution image is missing for the specified sample_id.
    """
    try:
        # Extract the RGB image from uns
        img_hires = adata.uns["spatial"][sample_id]["images"]["hires"]
    except KeyError as e:
        raise ValueError(f"Missing hires image for sample '{sample_id}'") from e

    # Convert from interleaved (H, W, C) to channel-first (C, H, W)
    img_arr = np.transpose(img_hires, (2, 0, 1))

    # Save image to OME-Zarr
    rgb_img_to_ome_zarr(
        img_arr,
        output_path,
        axes="cyx",              # color, y, x
        chunks=(1, 256, 256),    # default chunking
        img_name="H & E Image"   # name shown in Vitessce
    )


In [7]:
def make_nmf_adata(adata, sample_id, output_path, prefix="nmf"):
    """
    Create and save a new AnnData object containing NMF features 
    (obs['nmf1'] ... obs['nmf100']) as its expression matrix.

    Args:
        adata (AnnData): Input AnnData with nmf1...nmf100 in .obs.
        sample_id (str): Sample identifier.
        output_path (str): Path to save the resulting AnnData as zarr.
    """
    # Collect NMF columns
    nmf_cols = [col for col in adata.obs.columns if col.startswith(prefix)]
    nmf_matrix = adata.obs[nmf_cols].copy()

    # Replace NaNs with 0.0 and report
    nan_counts = nmf_matrix.isna().sum()
    total_nans = nan_counts.sum()
    if total_nans > 0:
        print(f"\t\t⚠️ Found {total_nans} NaNs across {nan_counts[nan_counts > 0].shape[0]} components.")
        print("\t\tPer-component NaN counts:")
        for idx, val in nan_counts[nan_counts > 0].items():
            print(f"\t\t\t{idx}: {val}")
        nmf_matrix = nmf_matrix.fillna(0.0)
        print("\t\t✅ Replaced all NaNs with 0.0")
    else:
        print("\t\tNo NaNs found in NMF features.")

    # Build new AnnData
    nmf_adata = sc.AnnData(
        X=nmf_matrix.to_numpy(),
        obs=adata.obs.copy(),
        obsm={k: adata.obsm[k] for k in ["spatial", "segmentations"] if k in adata.obsm},
        uns={"sample_id": sample_id}
    )
    nmf_adata.var_names = nmf_cols  # label the columns as nmf1...nmf100 (changes with prefix since they may not be called "nmf")

    # Optimize before saving
    optimized_nmf = optimize_adata(
        nmf_adata,
        obsm_keys=["spatial", "segmentations"],
        optimize_X=True,
        to_dense_X=True,
    )

    # Save
    print(f"\tSaving NMF AnnData to {output_path}")
    optimized_nmf.write_zarr(output_path, chunks=[optimized_nmf.shape[0], 10])


In [8]:
def make_vitessce_config(sample_id, base_output_dir, template_path):
    """
    Create a Vitessce config JSON for a given sample_id by 
    replacing `<<SAMPLE_NAME>>` in the template.

    Args:
        sample_id (str): Sample identifier to insert.
        base_output_dir (str): Path to the parent sample directory (contains 'data').
        template_path (str): Path to the template_config.json file.
    """
    # Load template as raw string
    with open(template_path, "r") as f:
        template_str = f.read()

    # Replace placeholder
    config_str = template_str.replace("<<SAMPLE_NAME>>", sample_id)

    # Make configs directory (sibling to data)
    configs_dir = os.path.join(base_output_dir, "configs")
    os.makedirs(configs_dir, exist_ok=True)

    # Save output config
    output_path = os.path.join(configs_dir, f"{sample_id}_config.json")
    print(f"\tSaving Vitessce config to {output_path}")
    with open(output_path, "w") as f:
        f.write(config_str)
    
    return output_path

In [ ]:
output_dir = "/zata/public_html/users/kresgeb/hippocampus/nmf_compare"
config_template_path = "/zata/zippy/kresgeb/hippocampus/my_output/nmf/HPC/HPC_nmf_compare_template_config.json"

for sample_id, adata in sample_dict.items():
    print(f"Processing {sample_id}")

    print("\tApplying scalefactor...")
    scale_spatial_coords(adata, sample_id)

    print("\tAdding segmentations...")
    add_segmentations(adata)

    print("\tOptimizing AnnData...")
    optimized_adata = optimize_adata(
        adata,
        obs_cols=["broad.domain", "domain"],
        obsm_keys=["spatial", "segmentations"],
        layer_keys=["logcounts"],
        optimize_X=True,
        to_dense_X=True,
    )

    # Paths for saving
    sample_output_dir = os.path.join(output_dir, "data", sample_id)
    os.makedirs(sample_output_dir, exist_ok=True)

    # Save optimized AnnData
    optimized_adata_path = os.path.join(sample_output_dir, "data.h5ad.zarr")
    print(f"\tSaving AnnData to {optimized_adata_path}")
    optimized_adata.write_zarr(optimized_adata_path, chunks=[optimized_adata.shape[0], 10])

    # Create and save NMF AnnData
    paper_nmf_adata_path = os.path.join(sample_output_dir, "paper_nmf_data.h5ad.zarr")
    print(f"\tCreating Paper NMF AnnData...")
    make_nmf_adata(adata, sample_id, paper_nmf_adata_path)

    my_nmf_adata_path = os.path.join(sample_output_dir, "my_nmf_data.h5ad.zarr")
    print(f"\tCreating My NMF AnnData...")
    make_nmf_adata(adata, sample_id, my_nmf_adata_path, prefix="my_nmf")

    # Save the image (OME-Zarr)
    image_output_path = os.path.join(sample_output_dir, "image.ome.zarr")
    print(f"\tSaving image to {image_output_path}")
    write_ome_zarr_image(adata, sample_id, image_output_path)

    # Create and save config json
    print(f"\tCreating sample config...")
    make_vitessce_config(sample_id, output_dir, config_template_path)

Processing V10B01-086_D1
	Applying scalefactor...
	Adding segmentations...
	Optimizing AnnData...


	Saving AnnData to /zata/public_html/users/kresgeb/hippocampus/nmf_compare/data/V10B01-086_D1/data.h5ad.zarr
	Creating Paper NMF AnnData...
		⚠️ Found 10167 NaNs across 3 components.
		Per-component NaN counts:
			nmf2: 3389
			nmf3: 3389
			nmf16: 3389
		✅ Replaced all NaNs with 0.0
	Saving NMF AnnData to /zata/public_html/users/kresgeb/hippocampus/nmf_compare/data/V10B01-086_D1/paper_nmf_data.h5ad.zarr
	Creating My NMF AnnData...
		⚠️ Found 10167 NaNs across 3 components.
		Per-component NaN counts:
			my_nmf2: 3389
			my_nmf3: 3389
			my_nmf16: 3389
		✅ Replaced all NaNs with 0.0
	Saving NMF AnnData to /zata/public_html/users/kresgeb/hippocampus/nmf_compare/data/V10B01-086_D1/my_nmf_data.h5ad.zarr
	Saving image to /zata/public_html/users/kresgeb/hippocampus/nmf_compare/data/V10B01-086_D1/image.ome.zarr
	Creating sample config...
	Saving Vitessce config to /zata/public_html/users/kresgeb/hippocampus/nmf_compare/configs/V10B01-086_D1_config.json
Processing V10B01-086_C1
	Applying scal

: 